In [1]:
# 07_geometric_factor.ipynb
# Reviewer 2 requests: (i) validate the closed form phi_m(0)=int rho+ rho- for
# the margin density; (ii) validate the migration closed form M=(sum rho(q_k))E|e|
# and E|delta|=sqrt(2)E|e|; (iii) show the separation ratio scales linearly in the
# predictor (K-1) rho_bar / phi_m(0); (iv) exhibit the validity window and the
# saturation of the linear-in-K law at K* ~ W / E|e|.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import gaussian_kde
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def fq(D): return b0 + (D * np.round(C / D)).sum(1)

fp, fn = f[y == 1], f[y == 0]
kp, kn, kall = gaussian_kde(fp), gaussian_kde(fn), gaussian_kde(f)
grid = np.linspace(f.min(), f.max(), 500); h = grid[1] - grid[0]

# (i) phi_m(0) closed form vs direct
phi_conv = float(np.sum(kp(grid) * kn(grid)) * h)
r = np.random.default_rng(0)
d = fp[r.integers(0, len(fp), 400000)] - fn[r.integers(0, len(fn), 400000)]
bw = 0.05; phi_direct = float(np.mean(np.abs(d) < bw / 2) / bw)

# (ii)-(iii) at Delta=0.1
D = 0.1; fqv = fq(D); e = fqv - f; Ee = float(np.abs(e).mean())
Edelta = float(np.abs(e[r.integers(0, len(e), 400000)] - e[r.integers(0, len(e), 400000)]).mean())
W = float(np.ptp(f))

def migration(fc, fqv, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1])
    return float(np.mean(np.digitize(fc, q) != np.digitize(fqv, q)))

def reversal(fc, fqv, npair=300000, seed=1):
    rr = np.random.default_rng(seed); a = rr.choice(np.where(y == 1)[0], npair); b = rr.choice(np.where(y == 0)[0], npair)
    return float(np.mean((np.sign(fc[a] - fc[b]) > 0) & (np.sign(fqv[a] - fqv[b]) <= 0)))

R = reversal(f, fqv)
Ks = [4, 5, 7, 10, 14, 20, 28, 40]
rows = []
for K in Ks:
    q = np.quantile(f, np.linspace(0, 1, K + 1)[1:-1])
    sum_rho = float(np.sum(kall(q)))
    rho_bar = sum_rho / (K - 1)
    M = migration(f, fqv, K)
    rows.append(dict(K=K, M=M, M_pred=sum_rho * Ee, predictor=(K - 1) * rho_bar / phi_conv,
                     ratio_obs=M / R))
tab = pd.DataFrame(rows)
tab.to_csv("../results/tables/07_geometric_factor.csv", index=False)
cfit = float(np.polyfit(tab.predictor, tab.ratio_obs, 1)[0])
print("phi_m(0): convolution=%.4f direct=%.4f" % (phi_conv, phi_direct))
print("E|e|=%.4f E|delta|=%.4f ratio=%.3f (sqrt2=%.3f)" % (Ee, Edelta, Edelta / Ee, np.sqrt(2)))
print("M closed form max rel. error = %.1f%%" % float((np.abs(tab.M - tab.M_pred) / tab.M).max() * 100))
print("separation ratio vs predictor: linear slope c = %.3f" % cfit)
print("W=%.3f  E|e|=%.4f  K* = W/E|e| = %.0f" % (W, Ee, W / Ee))

# (iv) saturation sweep in K at fixed Delta
Kbig = np.unique(np.round(np.logspace(np.log10(2), np.log10(200), 20)).astype(int))
Msat = np.array([migration(f, fqv, int(K)) for K in Kbig])
Kstar = W / Ee

fig, ax = plt.subplots(2, 2, figsize=(9, 7))
# (a) overlap integrand
ax[0, 0].plot(grid, kp(grid), ls="-", color="0.0", label=r"$\rho_+$")
ax[0, 0].plot(grid, kn(grid), ls="--", color="0.45", label=r"$\rho_-$")
ax[0, 0].fill_between(grid, kp(grid) * kn(grid) * 6, color="0.8", label=r"$6\,\rho_+\rho_-$ (overlap)")
ax[0, 0].set_xlabel("score"); ax[0, 0].set_ylabel("density"); ax[0, 0].legend(fontsize=7)
# (b) M closed form
ax[0, 1].plot(tab.M_pred, tab.M, "o", color="0.0")
lim = [0, float(max(tab.M.max(), tab.M_pred.max())) * 1.1]
ax[0, 1].plot(lim, lim, ls=":", color="0.5")
ax[0, 1].set_xlabel(r"predicted $(\sum_k\rho(q_k))\,E|e|$"); ax[0, 1].set_ylabel("observed migration"); ax[0, 1].legend(["cells", "$y=x$"], fontsize=7)
# (c) ratio linearity
ax[1, 0].plot(tab.predictor, tab.ratio_obs, "o", color="0.0", label="observed")
xs = np.linspace(0, tab.predictor.max(), 50)
ax[1, 0].plot(xs, cfit * xs, ls="--", color="0.45", label=f"slope {cfit:.2f}")
ax[1, 0].set_xlabel(r"$(K-1)\,\bar\rho/\phi_m(0)$"); ax[1, 0].set_ylabel("observed M/R"); ax[1, 0].legend(fontsize=7)
# (d) saturation
ax[1, 1].plot(Kbig, Msat, "-o", color="0.0", label="migration")
ax[1, 1].plot(Kbig, np.minimum(1, tab.M_pred.iloc[3] / (tab.K.iloc[3] - 1) * (Kbig - 1)), ls=":", color="0.5", label="linear law")
ax[1, 1].axvline(Kstar, ls="--", color="0.4", label=r"$K^*=W/E|e|$")
ax[1, 1].set_xscale("log"); ax[1, 1].set_xlabel("number of bands K"); ax[1, 1].set_ylabel("migration rate"); ax[1, 1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "07_geometric_factor"); plt.close(fig)

phi_m(0): convolution=0.0971 direct=0.0958
E|e|=0.0965 E|delta|=0.1368 ratio=1.418 (sqrt2=1.414)
M closed form max rel. error = 10.6%
separation ratio vs predictor: linear slope c = 0.973
W=9.006  E|e|=0.0965  K* = W/E|e| = 93
